# VLA-JEPA Few-Shot Training on Google Colab

This notebook trains a VLA-JEPA policy on your custom dataset with a frozen Qwen backbone (few-shot fine-tuning).

**What you need:**
- Hugging Face account and API token
- Dataset repo on HF Hub (e.g., `username/dataset_name`)
- Model repo name where to save trained policy (e.g., `username/my_vla_jepa_policy`)

## 1. Setup & Authentication

In [ ]:
import torch
import subprocess
import sys

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Training will be slow. Enable GPU in Runtime > Change runtime type")

In [ ]:
# Install LeRobot (this may take 2-3 minutes)
print("Installing LeRobot and dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lerobot[vla_jepa]"])
print("✓ Installation complete")

In [ ]:
from huggingface_hub import login
import getpass

# Login to Hugging Face
print("Please enter your Hugging Face API token.")
print("Get it from: https://huggingface.co/settings/tokens")
hf_token = getpass.getpass("HF Token: ")

login(token=hf_token)
print("✓ Logged in to Hugging Face Hub")

## 2. Configuration

In [ ]:
# Get user inputs
dataset_repo = input("Dataset repo ID (e.g., username/so101_pick_and_place_ring): ").strip()
output_repo = input("Output model repo ID (e.g., username/my_vla_jepa_policy): ").strip()
job_name = input("Job name (e.g., vla_jepa_so101_fewshot): ").strip() or "vla_jepa_fewshot"

print(f"\n📋 Configuration:")
print(f"  Dataset: {dataset_repo}")
print(f"  Output model: {output_repo}")
print(f"  Job name: {job_name}")

In [ ]:
# Training hyperparameters (optimized for few-shot with <50 episodes)
training_config = {
    "dataset_repo": dataset_repo,
    "policy_path": "lerobot/VLA-JEPA-Pretrain",  # Pretrained checkpoint
    "policy_repo": output_repo,
    "output_dir": "/tmp/vla_jepa_training",
    "job_name": job_name,
    
    # Few-shot specific settings
    "freeze_qwen": True,  # ← KEY: Freeze language backbone
    "reinit_modules": ["model.action_model.action_encoder", "model.action_model.action_decoder", "model.action_model.state_encoder"],  # For cross-embodiment
    "device": "cuda",
    
    # Training parameters
    "steps": 10000,        # ~3 epochs for ~50 episodes
    "batch_size": 4,       # Small batch for limited VRAM
    "num_workers": 4,
    "eval_freq": 1000,
    "save_freq": 500,
    "log_freq": 50,
    
    "wandb_enable": False,
    "seed": 1000,
}

print("\n⚙️  Training hyperparameters:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

## 3. Verify Dataset

In [ ]:
from datasets import load_dataset

print(f"Loading dataset: {training_config['dataset_repo']}...")
try:
    dataset = load_dataset(training_config['dataset_repo'])
    print(f"✓ Dataset loaded")
    print(f"\n📊 Dataset info:")
    for split, ds in dataset.items():
        print(f"  {split}: {len(ds)} samples")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print(f"Make sure:")
    print(f"  1. Repo exists: https://huggingface.co/{training_config['dataset_repo']}")
    print(f"  2. Your token has access (if private repo)")

## 4. Train VLA-JEPA

In [ ]:
import subprocess
import os
import json

# Build training command
train_cmd = [
    "lerobot-train",
    f"--dataset.repo_id={training_config['dataset_repo']}",
    f"--policy.path={training_config['policy_path']}",
    f"--policy.freeze_qwen={str(training_config['freeze_qwen']).lower()}",
    f"--policy.reinit_modules={json.dumps(training_config['reinit_modules'])}",
    f"--policy.device={training_config['device']}",
    f"--policy.repo_id={training_config['policy_repo']}",
    f"--output_dir={training_config['output_dir']}",
    f"--job_name={training_config['job_name']}",
    f"--steps={training_config['steps']}",
    f"--batch_size={training_config['batch_size']}",
    f"--num_workers={training_config['num_workers']}",
    f"--eval_freq={training_config['eval_freq']}",
    f"--save_freq={training_config['save_freq']}",
    f"--log_freq={training_config['log_freq']}",
    f"--seed={training_config['seed']}",
    f"--wandb.enable={str(training_config['wandb_enable']).lower()}",
]

print("🚀 Starting VLA-JEPA Few-Shot Training...\n")
print(f"Command: {' '.join(train_cmd)}\n")
print("="*80)

# Run training
try:
    result = subprocess.run(train_cmd, check=True, text=True)
    print("\n" + "="*80)
    print("✓ Training complete!")
except subprocess.CalledProcessError as e:
    print(f"\n❌ Training failed with error code {e.returncode}")
    print(f"Error: {e}")

## 5. Upload to Hugging Face Hub

In [ ]:
import os
from pathlib import Path

checkpoint_dir = Path(training_config['output_dir']) / "checkpoints" / "last" / "pretrained_model"

if checkpoint_dir.exists():
    print(f"✓ Found checkpoint at: {checkpoint_dir}")
    print(f"\n📤 The checkpoint should be automatically pushed to:")
    print(f"   https://huggingface.co/{training_config['policy_repo']}")
    print(f"\nIf it wasn't pushed automatically, you can push it with:")
    print(f"""\nfrom lerobot.policies import make_policy
policy = make_policy('{checkpoint_dir}')
policy.push_to_hub(repo_id='{training_config['policy_repo']}')
""")
else:
    print(f"❌ Checkpoint not found at: {checkpoint_dir}")
    print(f"\nLooking for checkpoints in {training_config['output_dir']}...")
    output_path = Path(training_config['output_dir'])
    if output_path.exists():
        for item in output_path.glob("**/pretrained_model"):
            print(f"  Found: {item}")
    else:
        print(f"  Output directory doesn't exist")

In [ ]:
# Verify model is on Hub
from huggingface_hub import model_info

try:
    info = model_info(training_config['policy_repo'])
    print(f"✓ Model found on Hub: {training_config['policy_repo']}")
    print(f"\nModel details:")
    print(f"  Last modified: {info.last_modified}")
    print(f"  Repo size: {info.siblings[0].size / 1e9:.1f} GB" if info.siblings else "  Size: unknown")
    print(f"\n🎉 Ready to use! Load with:")
    print(f"""\nfrom lerobot.policies import make_policy
policy = make_policy('{training_config['policy_repo']}')
""")
except Exception as e:
    print(f"⚠️  Model not found on Hub yet: {e}")
    print(f"It may still be uploading. Check: https://huggingface.co/{training_config['policy_repo']}")

## 6. Next Steps

### Evaluate on Your Robot

```bash
# On your local machine with the robot connected:
lerobot-rollout \
  --policy.path={output_repo} \
  --robot.type=so101_follower \
  --robot.port=/dev/ttyACM0 \
  --dataset.repo_id=username/rollout_eval
```

### Improve Performance

If the policy doesn't perform well:
1. **Collect more data**: Aim for 50-100 episodes (currently ~{len(dataset.get('train', []))} episodes)
2. **Longer training**: Increase `steps` to 20000-30000
3. **Adjust batch size**: If VRAM issues, reduce to 2-4

### Share Your Results

Add a README to your model repo:
```markdown
# VLA-JEPA Policy for [Task Name]

Fine-tuned from `lerobot/VLA-JEPA-Pretrain` on {training_config['dataset_repo']}

**Training config:**
- Frozen: Qwen backbone
- Steps: {training_config['steps']}
- Batch size: {training_config['batch_size']}
- Dataset episodes: ~[INSERT]
```